# BTSP vs BTSP_ELR: place-field potentiation on a treadmill

A treadmill loop with `n_positions` positions and `n_inputs` place-tuned input units, each with a Gaussian tuning curve (firing rate `x`) centred on one position. All inputs project onto **one output unit**, `out = relu(w · x)`.

- **Task 1:** a plateau at reward position `r1` on every lap (with probability `p_plateau`).
- **Task 2:** the reward moves to `r2`.

Each input has its own **eligibility trace** `ET`, kept separate from its firing rate. The trace decays slowly and is incremented by the input's rate at each position the animal passes:

$$ET \leftarrow \mathrm{clip}\left(ET \cdot e^{-1/\tau_{ET}} + et\_increment \cdot x(pos),\ 0,\ 1\right)$$

$\tau_{ET}$ is measured in positions. The trace carries over from lap to lap, because the treadmill is a loop. At the reward position, inputs that fired at earlier positions still have a large trace, so they are also potentiated.

The plateau `P` is a fixed amplitude at the reward position and 0 elsewhere. There is no error signal and no negative-plateau term.

**BTSP** (the positive-plateau part of `BTSP_CL.compute_delta_weight` in `EIANN/rules/btsp.py`):

$$\Delta w = \eta \, P \left[(w_{max} - w)\,ET - w \cdot dep\_ratio \cdot q_{dep}(ET)\right], \qquad w_{eq}(ET) = \frac{w_{max}\,ET}{ET + dep\_ratio \cdot q_{dep}(ET)}$$

**BTSP_ELR** (as in `BTSP_ELR_CL.step`): the same update scaled per weight by `lr_mod`, which is updated by

$$\Delta lr\_mod = P \cdot ET \cdot f(|w_{eq} - w|), \qquad lr\_mod \in [elr\_min, 1].$$

Here $f$ is piecewise linear, running from $-elr\_scale$ at $elr\_threshold$ to $+elr\_scale$ at $elr\_threshold + elr\_width$. Weights that sit near their equilibrium during plateaus lose plasticity, and weights far from it regain it.

All of the logic is self-contained in this notebook; nothing is imported from `EIANN`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['text.usetex'] = False
mpl.rcParams['axes.titlepad'] = 2.
mpl.rcParams['mathtext.default'] = 'regular'


def clean_axes(axes, left=True, right=False):
    '''
    Remove top and right axes from pyplot axes object.
    :param axes: list of pyplot.Axes
    :param left: bool
    :param right: bool
    '''
    if not type(axes) in [np.ndarray, list]:
        axes = [axes]
    elif type(axes) == np.ndarray:
        axes = axes.flatten()
    for axis in axes:
        axis.tick_params(direction='out')
        axis.spines['top'].set_visible(False)
        if not right:
            axis.spines['right'].set_visible(False)
        if not left:
            axis.spines['left'].set_visible(False)
        axis.get_xaxis().tick_bottom()
        axis.get_yaxis().tick_left()

## Helpers (numpy ports of `EIANN.utils.activations`)

In [ ]:
def scaled_rectified_sigmoid(th, peak, x=(0., 1.), ylim=(0., 1.)):
    '''
    Sigmoid rescaled to pass through ylim at the ends of x, with inputs clipped to x
    (port of get_scaled_rectified_sigmoid).
    :param th: float
    :param peak: float
    :param x: pair of float
    :param ylim: pair of float
    :return: callable
    '''
    slope = 2. / (peak - th)
    y = lambda xi: 1. / (1. + np.exp(-slope * (xi - th)))
    start_val = y(x[0])
    amp = y(x[-1]) - start_val
    target_amp = ylim[1] - ylim[0]
    return lambda xi: (target_amp / amp) * (y(np.clip(xi, x[0], x[-1])) - start_val) + ylim[0]


def pwlin(y0=0., y1=1., x0=0., x1=1.):
    '''
    y0 for x <= x0, y1 for x > x1, linear in between (port of pwlin).
    :return: callable
    '''
    slope = (y1 - y0) / (x1 - x0)
    def pwlin_instance(x):
        y = np.full_like(x, y0, dtype=float)
        y[x > x1] = y1
        indexes = (x0 < x) & (x <= x1)
        y[indexes] = slope * (x[indexes] - x0) + y0
        return y
    return pwlin_instance

## Environment: treadmill with place-tuned inputs

In [ ]:
def make_place_inputs(n_positions=10, n_inputs=10, sigma=1.5, peak=1., circular=True):
    '''
    Gaussian place tuning of each input unit, with centres evenly spaced along the track.
    :param n_positions: int
    :param n_inputs: int
    :param sigma: float; tuning width (in positions)
    :param peak: float; peak rate
    :param circular: bool; treat the treadmill as a loop
    :return: array (n_positions, n_inputs)
    '''
    positions = np.arange(n_positions)[:, None]
    centres = (np.arange(n_inputs) * n_positions / n_inputs)[None, :]
    d = np.abs(positions - centres)
    if circular:
        d = np.minimum(d, n_positions - d)
    return peak * np.exp(-d ** 2 / (2. * sigma ** 2))


def update_ET(ET, rate, tau_ET=2., et_increment=0.3):
    '''
    Decay the eligibility trace by one position, then increment it by the current input rates.
    :param ET: array (n_inputs,)
    :param rate: array (n_inputs,); input rates at the current position
    :param tau_ET: float; decay time constant (in positions)
    :param et_increment: float
    :return: array (n_inputs,); clipped to [0, 1]
    '''
    return np.clip(ET * np.exp(-1. / tau_ET) + et_increment * rate, 0., 1.)

## Learning rules

In [ ]:
class BTSP(object):
    def __init__(self, n_inputs, learning_rate=0.1, w_max=2., dep_ratio=1., dep_th=0.01, dep_width=0.01):
        '''
        BTSP update for positive plateaus (BTSP_CL.compute_delta_weight without the negative-plateau term).
        :param n_inputs: int
        :param learning_rate: float
        :param w_max: float
        :param dep_ratio: float
        :param dep_th: float
        :param dep_width: float
        '''
        self.n_inputs = n_inputs
        self.learning_rate = learning_rate
        self.w_max = w_max
        self.dep_ratio = dep_ratio
        self.q_dep = scaled_rectified_sigmoid(dep_th, dep_th + dep_width)
        self.lr_mod = np.ones(n_inputs)

    def direction(self, ET, w):
        return (self.w_max - w) * ET - w * self.dep_ratio * self.q_dep(ET)

    def w_eq(self, ET):
        w_eq = np.zeros_like(ET)
        valid = ET > 0.
        w_eq[valid] = self.w_max * ET[valid] / (ET[valid] + self.dep_ratio * self.q_dep(ET[valid]))
        return w_eq

    def step(self, w, ET, plateau):
        '''
        :param w: array (n_inputs,)
        :param ET: array (n_inputs,)
        :param plateau: float
        :return: array (n_inputs,); updated weights
        '''
        plateau_prob = max(plateau, 0.)
        delta_weight = plateau_prob * self.direction(ET, w)
        return np.clip(w + self.learning_rate * delta_weight, 0., self.w_max)


class BTSP_ELR(BTSP):
    def __init__(self, n_inputs, elr_threshold=0., elr_width=0.5, elr_scale=0.1, elr_min=0.01, **kwargs):
        '''
        BTSP with an elastic learning rate that depends on distance from w to w_eq (BTSP_ELR_CL.step without the
        negative-plateau term).
        :param elr_threshold: float
        :param elr_width: float
        :param elr_scale: float
        :param elr_min: float
        :param kwargs: passed to BTSP
        '''
        super().__init__(n_inputs, **kwargs)
        self.elr_threshold = elr_threshold
        self.elr_width = elr_width
        self.elr_scale = elr_scale
        self.elr_min = elr_min
        self.lr_mod_func = pwlin(-elr_scale, elr_scale, elr_threshold, elr_threshold + elr_width)

    def step(self, w, ET, plateau):
        plateau_prob = max(plateau, 0.)
        BTSP_delta_weight_direction = self.direction(ET, w)
        delta_weight = plateau_prob * BTSP_delta_weight_direction
        new_w = w + self.learning_rate * self.lr_mod * delta_weight

        # distance_to_w_eq = |w_eq - w| (pre-update w); inputs with ET == 0 get no lr_mod update
        denom = ET + self.dep_ratio * self.q_dep(ET)
        valid = ET > 0.
        distance_to_w_eq = -new_w.copy()
        distance_to_w_eq[valid] = np.abs(BTSP_delta_weight_direction[valid] / denom[valid])
        self.lr_mod = np.clip(self.lr_mod + plateau_prob * ET * self.lr_mod_func(distance_to_w_eq),
                              self.elr_min, 1.)
        return np.clip(new_w, 0., self.w_max)

## Simulation

In [ ]:
def run_tasks(rule, w0, inputs, tasks, tau_ET=2., et_increment=0.3, p_plateau=1., plateau_amp=1., relu_gate=False,
              seed=0):
    '''
    Run laps on the treadmill for a sequence of tasks. Positions are visited in order on each lap. At each position
    the eligibility traces are updated with the input rates and the output is computed from the current weights.
    Then a plateau is delivered at the reward position (with probability p_plateau) and the rule updates the weights.
    The traces start at 0 and are not reset between laps or tasks.
    :param rule: BTSP or BTSP_ELR
    :param w0: array (n_inputs,)
    :param inputs: array (n_positions, n_inputs); input rates at each position
    :param tasks: list of (reward_pos, n_laps)
    :param tau_ET: float; eligibility trace decay time constant (in positions)
    :param et_increment: float
    :param p_plateau: float; probability of a plateau at the reward position on each lap
    :param plateau_amp: float
    :param relu_gate: bool; only allow a plateau if the output is active
    :param seed: int
    :return: dict
    '''
    rng = np.random.default_rng(seed)
    n_positions = inputs.shape[0]
    w = np.array(w0, dtype=float)
    ET = np.zeros(inputs.shape[1])
    history = {'w': [], 'lr_mod': [], 'ET': [], 'w_lap': [], 'lr_mod_lap': [], 'output': [], 'plateau_events': [],
               'reward_pos': [], 'task_start_lap': [], 'w_task_end': []}
    lap = 0
    for reward_pos, n_laps in tasks:
        history['task_start_lap'].append(lap)
        for _ in range(n_laps):
            lap_output = np.zeros(n_positions)
            for pos in range(n_positions):
                rate = inputs[pos]
                ET = update_ET(ET, rate, tau_ET, et_increment)
                history['ET'].append(ET.copy())
                out = max(w @ rate, 0.)
                lap_output[pos] = out
                plateau = 0.
                if pos == reward_pos and rng.random() < p_plateau and (not relu_gate or out > 0.):
                    plateau = plateau_amp
                    history['plateau_events'].append((lap, pos))
                w = rule.step(w, ET, plateau)
                history['w'].append(w.copy())
                history['lr_mod'].append(rule.lr_mod.copy())
            history['output'].append(lap_output)
            history['w_lap'].append(w.copy())
            history['lr_mod_lap'].append(rule.lr_mod.copy())
            history['reward_pos'].append(reward_pos)
            lap += 1
        history['w_task_end'].append(w.copy())
    for key in ['w', 'lr_mod', 'ET', 'w_lap', 'lr_mod_lap', 'output', 'reward_pos', 'w_task_end']:
        history[key] = np.array(history[key])
    return history

## Parameters

In [ ]:
n_positions = 20
n_inputs = 20
sigma = 1.0
circular = True
tau_ET = 2.  # in positions
et_increment = 0.6

# moving the reward back to 1 recruits inputs 0-1, which were consolidated during task 1, so their lr_mod has to
# recover; inputs 2-3 (the task 1 field) have almost no ET at position 1 and stay consolidated
tasks = [(3, 20), (6, 20)]  # (reward position, number of laps)
p_plateau = 1.
plateau_amp = 1.
relu_gate = False
seed = 0

btsp_kwargs = dict(learning_rate=0.3, w_max=2., dep_ratio=1., dep_th=0.01, dep_width=0.01)
# lr_mod decreases when |w_eq - w| < elr_threshold + elr_width / 2 and increases when it is larger. These must be on
# the scale of the weights here (w_eq < ~1)
elr_kwargs = dict(elr_threshold=0.05, elr_width=0.2, elr_scale=0.3, elr_min=0.01)
w0 = np.full(n_inputs, 0.1 * btsp_kwargs['w_max'])

inputs = make_place_inputs(n_positions, n_inputs, sigma=sigma, circular=circular)

## Run

In [ ]:
def run_rules(inputs, tasks, w0, btsp_kwargs, elr_kwargs, **sim_kwargs):
    '''
    Run BTSP and BTSP_ELR from the same initial weights, with sanity checks.
    :param inputs: array (n_positions, n_inputs)
    :param tasks: list of (reward_pos, n_laps)
    :param w0: array (n_inputs,)
    :param btsp_kwargs: dict
    :param elr_kwargs: dict
    :param sim_kwargs: passed to run_tasks
    :return: (dict of rules, dict of histories)
    '''
    n_inputs = inputs.shape[1]
    rules = {'BTSP': BTSP(n_inputs, **btsp_kwargs),
             'BTSP_ELR': BTSP_ELR(n_inputs, **elr_kwargs, **btsp_kwargs)}
    results = {name: run_tasks(rule, w0, inputs, tasks, **sim_kwargs) for name, rule in rules.items()}

    for name, h in results.items():
        assert np.all((h['w'] >= 0.) & (h['w'] <= btsp_kwargs['w_max'])), name
    assert np.all((results['BTSP_ELR']['lr_mod'] >= elr_kwargs['elr_min']) & (results['BTSP_ELR']['lr_mod'] <= 1.))
    h_no_elr = run_tasks(BTSP_ELR(n_inputs, **{**elr_kwargs, 'elr_scale': 0.}, **btsp_kwargs), w0, inputs, tasks,
                         **sim_kwargs)
    assert np.allclose(h_no_elr['w'], results['BTSP']['w'])
    return rules, results


def get_ET_at_reward(h, tasks):
    '''
    ET at each task's reward position on the last lap of that task. The traces depend only on position, so this is the
    same for both rules.
    :param h: dict; history from run_tasks
    :param tasks: list of (reward_pos, n_laps)
    :return: list of array (n_inputs,)
    '''
    n_positions = h['output'].shape[1]
    task_end_laps = np.array(h['task_start_lap'][1:] + [len(h['output'])]) - 1
    return [h['ET'][lap * n_positions + reward_pos] for lap, (reward_pos, _) in zip(task_end_laps, tasks)]


sim_kwargs = dict(tau_ET=tau_ET, et_increment=et_increment, p_plateau=p_plateau, plateau_amp=plateau_amp,
                  relu_gate=relu_gate, seed=seed)
rules, results = run_rules(inputs, tasks, w0, btsp_kwargs, elr_kwargs, **sim_kwargs)
ET_at_reward = get_ET_at_reward(results['BTSP'], tasks)

## Plots

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.))
for i in range(n_inputs):
    ax.plot(np.arange(n_positions), inputs[:, i], label=f'input {i}')
for reward_pos, _ in tasks:
    ax.axvline(reward_pos, c='grey', ls='--', lw=1)
ax.set_xlabel('Position')
ax.set_ylabel('Input rate')
ax.set_title('Input place tuning')
clean_axes(ax)
fig.tight_layout()
plt.show()

In [ ]:
# ET of each input across positions on the last lap. The traces depend only on position, so they are the same for both
# rules and, after the first lap, on every lap
lap = len(results['BTSP']['output']) - 1
lap_ET = results['BTSP']['ET'][lap * n_positions:(lap + 1) * n_positions]
fig, ax = plt.subplots(figsize=(4.5, 3.))
for i in range(n_inputs):
    ax.plot(np.arange(n_positions), lap_ET[:, i], label=f'input {i}')
for reward_pos, _ in tasks:
    ax.axvline(reward_pos, c='grey', ls='--', lw=1)
ax.set_xlabel('Position')
ax.set_ylabel('ET')
ax.set_ylim(0., 1.)
ax.set_title('Eligibility traces')
clean_axes(ax)
fig.tight_layout()
plt.show()

In [ ]:
# final weights after each task, with the BTSP equilibrium weight for the ET at each task's reward position
fig, axes = plt.subplots(1, len(results), figsize=(4. * len(results), 3.), sharey=True)
for ax, (name, h) in zip(axes, results.items()):
    for t, (reward_pos, _) in enumerate(tasks):
        line, = ax.plot(np.arange(n_inputs), h['w_task_end'][t], 'o-', label=f'after task {t + 1}')
        ax.plot(np.arange(n_inputs), rules[name].w_eq(ET_at_reward[t]), '--', c=line.get_color(), alpha=0.6,
                label=f'$w_{{eq}}$ (reward at {reward_pos})')
    ax.plot(np.arange(n_inputs), w0, ':', c='k', label='$w_0$')
    ax.set_xlabel('Input unit')
    ax.set_title(name)
axes[0].set_ylabel('Weight')
axes[-1].legend(frameon=False, fontsize=8, bbox_to_anchor=(1., 1.))
clean_axes(axes)
fig.tight_layout()
plt.show()

In [ ]:
def plot_w_eq_mse(rules, results, tasks, title=None):
    '''
    Mean squared error over inputs between the weights and each task's w_eq, over train steps, for both rules. Each
    panel compares against one task's w_eq; dashed lines mark task boundaries.
    :param rules: dict; from run_rules
    :param results: dict; from run_rules
    :param tasks: list of (reward_pos, n_laps)
    :param title: str
    '''
    n_steps = len(results['BTSP']['w'])
    n_positions = results['BTSP']['output'].shape[1]
    task_start_steps = np.array(results['BTSP']['task_start_lap'] + [n_steps // n_positions]) * n_positions
    ET_at_reward = get_ET_at_reward(results['BTSP'], tasks)
    fig, axes = plt.subplots(1, len(tasks), figsize=(4.5 * len(tasks), 3.), sharey=True, squeeze=False)
    axes = axes[0]
    for t, (ax, ET, (reward_pos, _)) in enumerate(zip(axes, ET_at_reward, tasks)):
        for name, h in results.items():
            ax.plot(np.mean((h['w'] - rules[name].w_eq(ET)) ** 2, axis=1), label=name)
        for start in task_start_steps[1:-1]:
            ax.axvline(start - 0.5, c='grey', ls='--', lw=1)
        for k, (start, end) in enumerate(zip(task_start_steps[:-1], task_start_steps[1:])):
            ax.text((start + end) / 2., 1., f'task {k + 1}', transform=ax.get_xaxis_transform(), ha='center',
                    va='bottom', fontsize=8, c='grey')
        ax.set_xlabel('Train step')
        ax.set_title(f'vs task {t + 1} $w_{{eq}}$ (reward at {reward_pos})', pad=14)
    axes[0].set_ylabel('MSE(w, $w_{eq}$)')
    axes[-1].legend(frameon=False, fontsize=8)
    if title is not None:
        fig.suptitle(title)
    clean_axes(axes)
    fig.tight_layout()
    plt.show()


plot_w_eq_mse(rules, results, tasks)

In [ ]:
# movie of weights over train steps for both rules, with the current ET and the BTSP_ELR lr_mod
from matplotlib import animation
from IPython.display import HTML

mpl.rcParams['animation.embed_limit'] = 50.  # MB


def animate_weights(rules, results, tasks, w0, frame_stride=1, title=None):
    '''
    Movie of the weights of both rules over train steps (bars turn red on steps with a plateau), the w_eq of the
    current task, the current ET and the BTSP_ELR lr_mod. A vertical line marks the input centred on the current
    position.
    :param rules: dict; from run_rules
    :param results: dict; from run_rules
    :param tasks: list of (reward_pos, n_laps)
    :param w0: array (n_inputs,)
    :param frame_stride: int; show every frame_stride-th train step
    :param title: str; prefix for the frame titles
    :return: :class:'IPython.display.HTML'
    '''
    n_steps, n_inputs = results['BTSP']['w'].shape
    n_positions = results['BTSP']['output'].shape[1]
    frame_steps = np.arange(0, n_steps, frame_stride)
    task_start_lap = np.array(results['BTSP']['task_start_lap'])
    plateau_steps = {name: {lap * n_positions + pos for lap, pos in h['plateau_events']}
                     for name, h in results.items()}
    ET_at_reward = get_ET_at_reward(results['BTSP'], tasks)
    w_eq_by_task = {name: [rules[name].w_eq(ET) for ET in ET_at_reward] for name in results}
    # x location of the input unit centred on each position, to mark the animal's position
    pos_to_x = np.arange(n_positions) * n_inputs / n_positions

    x = np.arange(n_inputs)
    w_ylim = 1.1 * max(h['w'].max() for h in results.values())
    fig, axes = plt.subplots(2, 2, figsize=(8., 5.), sharex=True)
    weight_bars, w_eq_lines = {}, {}
    for ax, name in zip(axes[0], results):
        weight_bars[name] = ax.bar(x, results[name]['w'][0], color='C0')
        w_eq_lines[name], = ax.plot(x, w_eq_by_task[name][0], '--', c='grey', label='$w_{eq}$ (current task)')
        ax.plot(x, w0, ':', c='k', label='$w_0$')
        ax.set_ylim(0., w_ylim)
        ax.set_title(name)
    axes[0][0].set_ylabel('Weight')
    axes[0][1].legend(frameon=False, fontsize=8, loc='upper right')
    ET_bars = axes[1][0].bar(x, results['BTSP']['ET'][0], color='C2')
    axes[1][0].set_ylim(0., 1.)
    axes[1][0].set_ylabel('ET')
    lr_mod_bars = axes[1][1].bar(x, results['BTSP_ELR']['lr_mod'][0], color='C1')
    axes[1][1].set_ylim(0., 1.05)
    axes[1][1].set_ylabel('BTSP_ELR lr_mod')
    for ax in axes[1]:
        ax.set_xlabel('Input unit')
        ax.set_xticks(x)
    pos_lines = [ax.axvline(pos_to_x[0], c='k', lw=1, alpha=0.3) for ax in axes.flat]
    clean_axes(axes)
    fig.tight_layout(rect=(0., 0., 1., 0.94))

    def update(i):
        step = frame_steps[i]
        lap, pos = divmod(step, n_positions)
        t = np.searchsorted(task_start_lap, lap, side='right') - 1
        for name, h in results.items():
            color = 'C3' if step in plateau_steps[name] else 'C0'
            for bar, height in zip(weight_bars[name], h['w'][step]):
                bar.set_height(height)
                bar.set_color(color)
            w_eq_lines[name].set_ydata(w_eq_by_task[name][t])
        for bar, height in zip(ET_bars, results['BTSP']['ET'][step]):
            bar.set_height(height)
        for bar, height in zip(lr_mod_bars, results['BTSP_ELR']['lr_mod'][step]):
            bar.set_height(height)
        for line in pos_lines:
            line.set_xdata([pos_to_x[pos]] * 2)
        frame_title = f'Task {t + 1} (reward at {tasks[t][0]}), lap {lap}, position {pos}, step {step}'
        fig.suptitle(frame_title if title is None else f'{title}: {frame_title}')

    anim = animation.FuncAnimation(fig, update, frames=len(frame_steps), interval=100)
    plt.close(fig)
    # anim.save('BTSP_vs_BTSP_ELR_weights.gif', writer='pillow', fps=10)
    return HTML(anim.to_jshtml())


animate_weights(rules, results, tasks, w0)

In [ ]:
# a second animation with different parameters: edit any of the settings below
tasks_2 = [(4, 20), (14, 20)]
rules_2, results_2 = run_rules(inputs, tasks_2, w0, btsp_kwargs, elr_kwargs, **sim_kwargs)
title_2 = 'Reward ' + ' then '.join(str(reward_pos) for reward_pos, _ in tasks_2)
plot_w_eq_mse(rules_2, results_2, tasks_2, title=title_2)

In [ ]:
animate_weights(rules_2, results_2, tasks_2, w0, title=title_2)